# LangChain Agent Runtime Context — Fundamentals and Practical Example

This notebook focuses on **Context** in LangChain agents, using the account-information example.

## Learning
By the end, you should understand:

1. What runtime context is and why agents need it.
2. What `context_schema` does in `create_agent`.
3. How to define context with a Python class (`dataclass`).
4. How to pass a context object through `agent.invoke(..., context=...)`.
5. How a tool reads it through `runtime.context`.
6. The difference between **context**, **state**, **thread_id/config**, and **store**.
7. Why context is passed on every invocation, even when a checkpointer is configured.

**Important:** The `USER_DATABASE` below is a simple mock database for the demonstration. It is a Python dictionary, but it is not the LangChain context. The context is the `UserContext` class instance passed to `agent.invoke()`.

## 1. The mental model: four different things

| Concept | What it represents | Typical lifetime | Example |
|---|---|---|---|
| Context | Immutable, run-specific application data | One invocation | `user_id`, tenant, permissions |
| State | Mutable conversation data | A conversation/thread | messages, workflow status |
| `thread_id` in config | Identifies which conversation/checkpoint to resume | Across invocations | `"conversation-123"` |
| Store | Long-term data shared across threads | Across conversations | saved preferences, user memories |

### A useful way to remember it

- **Context answers:** Who is making this run, and what trusted settings apply?
- **State answers:** What has happened in this conversation?
- **Thread ID answers:** Which conversation should be resumed?
- **Store answers:** What information should remain available across conversations?

Context itself is not the checkpointer and is not automatically persisted as conversation state. Pass the context object on each invocation that needs it.

## 2. How context flows through an agent

1. Your application creates a context object, for example `UserContext(user_id="user123")`.
2. `context_schema=UserContext` tells the agent what type of context it expects.
3. Your application passes the object with `context=context` on `agent.invoke(...)`.
4. LangChain makes the run context available to tools through `ToolRuntime`.
5. The tool reads the value with `runtime.context.user_id`.

The `runtime` argument is injected by LangChain. It is not an argument the model needs to generate or provide in its tool call.

## 3. Define context with a Python class

Yes — use a Python class rather than manually building a context dictionary.

For this example, use a frozen dataclass:

- `@dataclass` creates the class's constructor and related methods.
- `frozen=True` prevents ordinary reassignment of fields after construction.
- `user_id: str` declares the identity value the tool needs.

```python
from dataclasses import dataclass

@dataclass(frozen=True)
class UserContext:
    user_id: str
```

Then create a normal Python object:

```python
context = UserContext(user_id="user123")
```

This is a Python object, not a JSON string or dictionary. LangChain's documented context example uses a dataclass in this way.

## 4. The mock user database

This dictionary represents a stand-in for a real database. The tool uses the `user_id` from context to select the correct account.

In production, replace this dictionary lookup with a database or service call. Do not treat a user ID supplied directly in an untrusted request as proof of identity; derive trusted identity from your authenticated application layer.

In [6]:
USER_DATABASE = {
    "user123": {
        "name": "Alice Johnson",
        "account_type": "Premium",
        "balance": 5000,
        "email": "alice@example.com",
    },
    "user456": {
        "name": "Bob Smith",
        "account_type": "Standard",
        "balance": 1200,
        "email": "bob@example.com",
    },
}

print(USER_DATABASE["user123"])

{'name': 'Alice Johnson', 'account_type': 'Premium', 'balance': 5000, 'email': 'alice@example.com'}


## 5. Define the context class

This is the context schema for the agent.

Notice that this is separate from the database above:

- `USER_DATABASE` contains account records.
- `UserContext` describes the trusted invocation data passed into the agent.

We use `frozen=True` because context is intended to be immutable configuration for the run.

In [7]:
from dataclasses import dataclass

@dataclass(frozen=True)
class UserContext:
    user_id: str

# Construct a Python object — not a JSON string or dict.
context = UserContext(user_id="user123")

print(context)
print(context.user_id)

UserContext(user_id='user123')
user123


## 6. Access context inside a tool with `ToolRuntime`

The tool does not ask the model for a `user_id` argument. Instead, LangChain injects `ToolRuntime[UserContext]`, and the tool reads `runtime.context.user_id`.

This is useful when the application already knows the authenticated user and the model should not choose or modify that identity.

The tool returns only the account holder, account type, and balance for this teaching example. Avoid exposing sensitive account details unless the caller is authorized.

In [8]:
from langchain.tools import tool, ToolRuntime

@tool
def get_account_info(runtime: ToolRuntime[UserContext]) -> str:
    """Get the current authenticated user's account information."""

    # Read trusted invocation context
    user_id = runtime.context.user_id

    # Look up that user in the mock database
    user = USER_DATABASE.get(user_id)

    if user is None:
        return "User not found."

    return (
        f"Account holder: {user['name']}\n"
        f"Account type: {user['account_type']}\n"
        f"Balance: ${user['balance']}"
    )

## 7. Create the agent and declare `context_schema`

`context_schema=UserContext` tells the agent which context type will be supplied at invocation time.

`checkpointer=InMemorySaver()` enables short-term conversation persistence for a thread. It does **not** mean the context object is saved into the checkpoint.

The model is set to `gpt-5.5`, matching the current LangChain documentation example. You need a valid `OPENAI_API_KEY` in your environment.

In [9]:

from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.utils.uuid import uuid7


model = ChatOpenAI(model="gpt-5.5")

checkpointer = InMemorySaver()

agent = create_agent(
    model=model,
    tools=[get_account_info],
    context_schema=UserContext,
    checkpointer=checkpointer,
    system_prompt=(
        "You are an account information assistant. "
        "Use get_account_info when the user asks about their account. "
        "Never guess account details."
    ),
)

# A thread identifies the conversation to resume.
thread_id = str(uuid7())
config = {"configurable": {"thread_id": thread_id}}

print("Thread ID:", thread_id)

Thread ID: 01a1ba92-94f9-71f0-b09d-17c308c224fd


## 8. First invocation — pass both `config` and `context`

This is the first run in the conversation.

- `config` contains the stable `thread_id`.
- `context` contains the trusted `UserContext` object for this run.
- `messages` contains the user's conversational request.

The tool gets `user123` from `runtime.context`, looks up Alice's record, and returns the account information.

In [10]:
context = UserContext(user_id="user123")

result = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "What's my current balance?"}
        ]
    },
    config=config,
    context=context,
)

print(result["messages"][-1].content)

Your current balance is **$5,000**.


## 9. Second invocation — same thread, context passed again

This is a new invocation in the same conversation:

- Reuse the same `thread_id` so the checkpointer resumes the conversation.
- Pass `context=context` again because context is invocation-time data.
- The tool can access the same user identity for this run.

The previous conversation messages are restored by the checkpointer. Context is supplied again by the application.

In [ ]:
result = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "What was my balance again?"}
        ]
    },
    config=config,
    context=context,
)

print(result["messages"][-1].content)

## 10. Same agent, different user context

Context can vary between invocations. For example, the application can create a context object for Bob.

For a real application, use a separate thread per user's conversation. This cell uses a new thread for Bob to keep the conversation boundary clear.

The application — not the LLM — chooses the trusted user identity.

In [ ]:
bob_context = UserContext(user_id="user456")

bob_config = {
    "configurable": {
        "thread_id": str(uuid7())
    }
}

bob_result = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "What's my current balance?"}
        ]
    },
    config=bob_config,
    context=bob_context,
)

print(bob_result["messages"][-1].content)

## 11. Can we use Pydantic instead of a dataclass?

The documented example uses a dataclass, and that is a clear choice for simple context objects.

If your application already standardizes on Pydantic, you can express the context as a Pydantic model and pass an instance as the context object. For example:

```python
from pydantic import BaseModel, ConfigDict

class PydanticUserContext(BaseModel):
    model_config = ConfigDict(frozen=True)
    user_id: str

pydantic_context = PydanticUserContext(user_id="user123")
```

Use the matching context type consistently: set `context_schema=PydanticUserContext`, annotate the tool as `ToolRuntime[PydanticUserContext]`, and pass `context=pydantic_context`.

For this notebook, the main implementation uses the dataclass form from the official documentation.

## 12. Common misconceptions

### Is context the same as state?
No. Context is immutable run-specific configuration; state is mutable conversation data managed by the agent.

### Does the checkpointer save context?
Do not rely on that. The checkpointer persists thread state/checkpoints. Pass the context object on each invocation that needs it.

### Is `thread_id` the user ID?
Not necessarily. `thread_id` identifies a conversation. A user can have multiple conversations, so a stable user identity and a conversation ID serve different purposes.

### Does the model see `runtime` or choose `user_id`?
No. `runtime` is injected by LangChain and hidden from the model's tool schema. The application supplies the context.

### Is a Python dictionary database the same as context?
No. In this example, the dictionary is mock account storage. Context carries the identity key used to find a record.

### Is context a place to store secrets?
Avoid passing secrets unnecessarily. Keep credentials in a secret manager or secure server-side configuration, and pass only the minimum trusted information tools need.

## 13. Final execution flow

```text
Application / API
    |
    |-- creates UserContext(user_id="user123")
    |
    |-- agent.invoke(input, config=thread_config, context=context)
              |
              |-- config.thread_id -> selects conversation checkpoint
              |
              |-- context -> available for this invocation
              |
              v
          LangChain Agent
              |
              v
          get_account_info tool
              |
              |-- runtime.context.user_id
              |
              v
          USER_DATABASE lookup
              |
              v
          Account information returned
```

## Key takeaways

- Define context with a typed Python class (`dataclass` or a supported schema class).
- Register it with `context_schema=...`.
- Create an instance in your application.
- Pass it as `context=...` on every invocation that needs it.
- Read it inside tools through `runtime.context`.
- Use `thread_id` plus a checkpointer for conversation persistence.
- Keep identity and authorization decisions in trusted application code, not in model-generated tool arguments.

**Official references**
- [LangChain Tools — Access context](https://docs.langchain.com/oss/python/langchain/tools#access-context)
- [LangChain Short-term memory](https://docs.langchain.com/oss/python/langchain/short-term-memory)